# C2 Phase 1 v2 — NP50 with missing-execution Baseline retention

Amendment Plan SHA `3e99c78855ba09b6be1a95fb13318b35d551d473`. Original stopped Result `96` and outputs are preserved. Primary is unchanged: 50% planned duration, completed-M1 MFE < +0.25R. Missing exact-through-+4-minute execution retains full R0 with zero delta and remains in the denominator. NP75 is robustness. Exclude the same two anchors from both sides only in the predeclared descriptive sensitivity analysis.

2022–2026 is previously viewed, not a pristine holdout. No money simulation or operational changes.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil
import pandas as pd
from IPython.display import display
REPO=Path('/content/time-entry-portfolio-lab')
BASELINE=Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT=Path('/content/m1')
OUT=Path('/content/c2_no_progress_phase1_v2')
SAVE_TO_DRIVE=False
DRIVE_DEST=Path('/content/drive/MyDrive/time-entry-portfolio-lab/c2_no_progress_phase1_v2')
assert REPO.is_dir() and BASELINE.is_file() and M1_ROOT.is_dir()
record=REPO/'results/c2_no_progress_phase1_v2/c2_no_progress_phase1_run_record.csv'
IMPLEMENTATION_SHA=str(pd.read_csv(record).iloc[0]['ImplementationSHA']) if record.exists() else subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
for relative in ['src/research/c2_no_progress_phase1_v2.py','src/research/c2_no_progress_phase1.py','tests/verify_c2_no_progress_phase1_v2.py','src/research/c1_path_management_phase1.py','src/research/exit_efficiency_phase1.py','src/research/daily_stop_baseline_revalidation.py']:
    frozen=subprocess.check_output(['git','-C',str(REPO),'show',f'{IMPLEMENTATION_SHA}:{relative}'])
    assert (REPO/relative).read_bytes()==frozen, relative
print('Frozen implementation:',IMPLEMENTATION_SHA)


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(REPO/'tests'),'-p','test_c2_no_progress_phase1*.py','-v'],check=True)
args=['--baseline',str(BASELINE),'--manifest',str(REPO/'results/volatility_phase1/volatility_phase1_input_manifest.csv'),'--m1-root',str(M1_ROOT),'--out',str(OUT)]
subprocess.run([sys.executable,str(REPO/'src/research/c2_no_progress_phase1_v2.py'),*args,'--implementation-sha',IMPLEMENTATION_SHA],check=True)
subprocess.run([sys.executable,str(REPO/'tests/verify_c2_no_progress_phase1_v2.py'),*args],check=True)
def table(name): return pd.read_csv(OUT/f'c2_no_progress_phase1_{name}.csv')


## 1. Source / Hash Audit

In [ ]:
display(table('m1_audit'))

## 2. R0 Reconciliation

In [ ]:
display(table('r0_reconciliation'))

## 3. NP50 Trigger Coverage

In [ ]:
display(table('coverage'))

## Missing-execution policy audit

In [ ]:
display(table('missing_execution_policy_audit'))

## 4. Portfolio R0 vs NP50

In [ ]:
display(table('portfolio_summary'))

## 5. Historical / Recent Results

In [ ]:
display(table('period_summary'))

## 6. Paired Delta CI

In [ ]:
display(table('trade_delta_summary'))

## 7. Triggered Trades

In [ ]:
display(table('trigger_summary'))

## 8. Recovery Diagnostic

In [ ]:
display(table('recovery_summary'))

## 9. NP75 Robustness

In [ ]:
display(table('np75_robustness'))

## 10. Strategy Diagnostics

In [ ]:
display(table('strategy_summary'))

## 11. Formal A–H

In [ ]:
display(table('formal_gates'))

## Excluding two anchors from both sides — descriptive sensitivity

In [ ]:
display(table('sensitivity_period_summary'))

## Sensitivity gates — descriptive only

In [ ]:
display(table('sensitivity_gates'))

## 12. Verdict / 13. Phase 2 eligibility

In [ ]:
display(table('run_record'))

## Independent verification

In [ ]:
display(table('independent_verification'))

## Representative M1 checks

In [ ]:
display(table('manual_audit'))

Recovery flags overlap. NEVER_RECOVERED means baseline final R <=0, not absence of a temporary recovery. Safety uses entry-day/week sums and cumulative realized-R drawdown. Strategy positives remain exploratory only. No Phase 2 or checkpoint/threshold tuning is performed.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DEST.mkdir(parents=True,exist_ok=True)
    for file in OUT.glob('*.csv'): shutil.copy2(file,DRIVE_DEST/file.name)
else:
    print('Drive save OFF; results in',OUT)
